In [2]:
!pip install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 79.0 MB/s  0:00:006m0:00:01

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
from pulp import *

# Load data
warehouses = pd.read_csv('../data/raw/warehouses.csv')
warehouses = warehouses.set_index('warehouse_id')
stores = pd.read_csv('../data/raw/stores.csv', index_col='city')
distances = pd.read_csv('../data/raw/distance_matrix.csv')
forecasts = pd.read_csv('../data/processed/future_forecasts.csv')

In [ ]:
jan_2025 = forecasts[forecasts['ds'] == '2025-01-01'][['city', 'yhat']].copy()
jan_2025.columns = ['city', 'demand']
jan_2025['demand'] = jan_2025['demand'].round(0).astype(int)

stores_reset = stores.reset_index()[['store_id', 'city']]
jan_2025 = jan_2025.merge(stores_reset, on='city')
jan_2025 = jan_2025.set_index('store_id')

print("Store demand (Jan 2025):")
print(jan_2025[['city', 'demand']])
print(f"\nTotal demand: {jan_2025['demand'].sum():,} tonnes")
print(f"Total warehouse capacity: {warehouses['capacity_tonnes'].sum():,} tonnes")

Store demand (Jan 2025):
                 city  demand
store_id                     
S001       Casablanca    4566
S002            Rabat    1728
S003              Fès    4142
S004        Marrakech    3211
S005           Agadir    1941
S006           Tanger    3210
S007            Oujda    1374
S008         Laâyoune     648
S009           Meknès    2108
S010      Béni Mellal    1959
S011          Kénitra    2309
S012          Tétouan    1958
S013             Safi    1252
S014        El Jadida    1048
S015            Nador    1038
S016           Settat    1026
S017        Khouribga    1401
S018        Essaouira     946
S019       Ouarzazate    1053
S020           Dakhla     782

Total demand: 37,700 tonnes
Total warehouse capacity: 58,500 tonnes


In [ ]:
cost_matrix = distances[['warehouse_id', 'store_id', 'distance_km']].copy()
cost_matrix['cost_per_tonne'] = cost_matrix['distance_km'] * 0.35
cost_matrix = cost_matrix.pivot(
    index='warehouse_id',
    columns='store_id',
    values='cost_per_tonne'
)
print(cost_matrix.round(1))

store_id       S001   S002   S003   S004   S005   S006   S007   S008   S009  \
warehouse_id                                                                  
W1              0.0   29.8   85.8   76.5  139.0  101.9  188.4  312.4   67.4   
W2             29.8    0.0   59.5  100.1  166.5   74.8  160.9  341.2   42.2   
W3             85.8   59.5    0.0  135.1  206.3   72.1  102.7  383.0   18.5   
W4             76.5  100.1  135.1    0.0   71.4  174.8  230.9  247.9  118.7   
W5            139.0  166.5  206.3   71.4    0.0  240.7  301.8  176.7  189.6   
W6            101.9   74.8   72.1  174.8  240.7    0.0  131.9  413.9   73.0   
W7            188.4  160.9  102.7  230.9  301.8  131.9    0.0  476.7  121.2   
W8            312.4  341.2  383.0  247.9  176.7  413.9  476.7    0.0  366.3   

store_id       S010   S011   S012   S013   S014   S015   S016   S017   S018  \
warehouse_id                                                                  
W1             62.8   42.2  105.6   73.2   32.5  16

In [5]:
print("store_ids:", store_ids[:5])
print("cost_matrix columns:", cost_matrix.columns.tolist()[:5])
print("warehouse_ids:", warehouse_ids[:3])
print("cost_matrix index:", cost_matrix.index.tolist())


store_ids: ['S001', 'S002', 'S003', 'S004', 'S005']
cost_matrix columns: ['S001', 'S002', 'S003', 'S004', 'S005']
warehouse_ids: ['Casablanca', 'Rabat', 'Fès']
cost_matrix index: ['W1', 'W2', 'W3', 'W4', 'W5', 'W6', 'W7', 'W8']


In [7]:
prob = LpProblem("Morocco_Supply_Chain_Optimization", LpMinimize)

warehouse_ids = warehouses.index.tolist()
store_ids = jan_2025.index.tolist()

x = LpVariable.dicts(
    "ship",
    [(w, s) for w in warehouse_ids for s in store_ids],
    lowBound=0,
    cat='Continuous'
)

prob += lpSum(
    x[(w, s)] * cost_matrix.loc[w, s]
    for w in warehouse_ids
    for s in store_ids
)

for s in store_ids:
    prob += lpSum(x[(w, s)] for w in warehouse_ids) == jan_2025.loc[s, 'demand']

for w in warehouse_ids:
    prob += lpSum(x[(w, s)] for s in store_ids) <= warehouses.loc[w, 'capacity_tonnes']

prob.solve(PULP_CBC_CMD(msg=0))

print(f"Status: {LpStatus[prob.status]}")
print(f"Total transport cost: {value(prob.objective):,.0f} MAD/month")

Status: Optimal
Total transport cost: 640,293 MAD/month


In [10]:
allocation = []
for w in warehouse_ids:
    for s in store_ids:
        qty = value(x[(w, s)])
        if qty > 0:
            allocation.append({
                'warehouse_id': w,
                'store_id': s,
                'city_store': jan_2025.loc[s, 'city'],
                'city_warehouse': warehouses.loc[w, 'city'],
                'quantity_tonnes': round(qty, 1),
                'transport_cost_mad': round(qty * cost_matrix.loc[w, s], 0)
            })

allocation_df = pd.DataFrame(allocation)
allocation_df = allocation_df.sort_values('transport_cost_mad', ascending=False)

print(f"Active routes: {len(allocation_df)}")
print(f"Total cost: {allocation_df['transport_cost_mad'].sum():,.0f} MAD")
print(allocation_df)

Active routes: 20
Total cost: 640,293 MAD
   warehouse_id store_id   city_store city_warehouse  quantity_tonnes  \
19           W8     S020       Dakhla       Laâyoune            782.0   
9            W4     S010  Béni Mellal      Marrakech           1959.0   
10           W4     S013         Safi      Marrakech           1252.0   
3            W1     S017    Khouribga     Casablanca           1401.0   
11           W4     S019   Ouarzazate      Marrakech           1053.0   
13           W5     S018    Essaouira         Agadir            946.0   
17           W7     S015        Nador          Oujda           1038.0   
7            W3     S009       Meknès            Fès           2108.0   
1            W1     S014    El Jadida     Casablanca           1048.0   
15           W6     S012      Tétouan         Tanger           1958.0   
5            W2     S011      Kénitra          Rabat           2309.0   
2            W1     S016       Settat     Casablanca           1026.0   
0        

In [11]:
allocation_df.to_csv('../data/processed/optimized_allocation.csv', index=False)
print("Saved optimized allocation")

Saved optimized allocation


## Optimization Results — January 2025

| Metric | Value |
|---|---|
| Baseline transport cost | 1,800,514 MAD/month |
| Optimized transport cost | 640,293 MAD/month |
| Monthly saving | 1,160,221 MAD/month |
| Annual saving | 13,922,652 MAD/year |
| Saving percentage | 64% |
| Active routes | 20 (one per store) |
| Status | Optimal (mathematically proven) |